# 延迟可用的季节性盈利特征：固定训练回测
日期缓冲是研究假设，不是首次公布时钟认证。单数值特征、共同可用标记、6拟合6完整账户，强参考及SOXX均保留。

In [1]:
from pathlib import Path
from dataclasses import asdict
from copy import deepcopy
import os,json,hashlib
import numpy as np,pandas as pd,torch,qlib
from qlib.workflow import R
from src.workflow.minute_action_value import _source
from src.workflow.minute_pit_allocator import _dataset,_auxiliary_dataset,_prediction_dates,_ensemble
from src.model.minute_pit_allocator import TARGETS
from src.model.tangent_allocator import TangentHeadAllocator,audit_zero_extended_heads
from src.data.delayed_seasonal_eps import delayed_seasonal_events,align_delayed_events,FEATURE,FLAG
from src.data import QlibProviderConfig,build_qlib_store_from_frame,load_qlib_feature_frame
from qlib.data import D
from src.workflow.pit_constituent_auxiliary_training_allocation import _smooth_weights
from src.workflow.stage1_open_action_value import Stage1OpenActionValueSpec
from src.workflow.intraday_recovery_direct_allocation import IntradayRecoveryDirectAllocationSpec
from src.backtest.full_path_tiered import replay_full_path
from src.backtest.stage1_ibkr import native_tiered_stage1
from src.backtest.qlib_continuous_position import QlibContinuousPosition
from src.analysis.integrated_allocation import paired_metrics,annual_paired_metrics,stationary_bootstrap_excess
from src.data.minute_intake import atomic_json
from src.data.nas_market_data import archive_nas_recorder
root=Path('/Users/rziiiii/dev/workspace');card=root/'research/delayed_seasonal_eps_training_20261015';p=json.loads((card/'plan.json').read_text());provider=root/'data/qlib/full_path_tiered_allocation_20261013'
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(provider),region='us')
assert hashlib.sha256((root/'production/stage1_futu_sim_v1/runtime_release.json').read_bytes()).hexdigest()==p['release_sha256']
training,ta=_source(root,p['training_source'],['inputs/daily_frame','inputs/features','inputs/common','inputs/timing','inputs/auxiliary','inputs/audit',*[f'models/base_outer_{seed}' for seed in p['seeds']]])
source,sa=_source(root,p['source'],['inputs/frame']);controller,ca=_source(root,p['price_source'],['inputs/controller','inputs/plan']);action,aa=_source(root,p['action_source'],['inputs/plan','inputs/market','controller/source_recovery_selection']);strong,sta=_source(root,p['strong_source'],['inputs/plan',*[f'analysis/native/{family}_{cost}' for family in ['candidate','control','soxx'] for cost in p['slippage_bps']]])
for key in ['initial_cash','tariff','start','end','as_of','slippage_bps','maximum_soxl_weight','minimum_cash_weight','maximum_drawdown']:assert strong['inputs/plan'][key]==p[key]
assert controller['inputs/plan']['tariff']==p['tariff']
original=training['models/base_outer_71'];base=training['inputs/daily_frame']
eps,ea=_source(root,p['eps_source'],['analysis/verified_facts']);splits,spa=_source(root,p['split_source'],['inputs/split_ledger']);assert p['source_gate'].startswith('Conditional')
qlib.init(provider_uri=str(root/'data/qlib/minute_information_pit_allocator_20261012/daily_prices'),region='us');calendar=pd.DatetimeIndex(D.calendar(start_time='2010-03-11',end_time=p['as_of'],freq='day'));events=delayed_seasonal_events(eps['analysis/verified_facts'],splits['inputs/split_ledger'],calendar);features=align_delayed_events(events,base.index,base.trade_date)
feature_provider=root/'data/qlib/delayed_seasonal_eps_20261015';payload=features.copy();payload.insert(0,'datetime',features.index);payload['instrument']='SOXL_SOXX_CASH';payload=payload.reset_index(drop=True);manifest=build_qlib_store_from_frame(payload,provider_uri=feature_provider,freq='day',region='us',as_of=p['as_of'],field_map={'$'+k:k for k in features},source='48 original-accession quarterly EPS text crosschecks; conditional date-buffered research; source identities in frozen plan')
read=load_qlib_feature_frame(QlibProviderConfig(feature_provider),instruments=['SOXL_SOXX_CASH'],fields=['$'+FEATURE,'$'+FLAG],start=str(base.index.min().date()),end=p['as_of']).xs('SOXL_SOXX_CASH',level='instrument').rename(columns=lambda x:x.removeprefix('$')).astype(float);assert read.index.equals(features.index);np.testing.assert_allclose(read,features,rtol=1e-6,atol=1e-7,equal_nan=True)
features=read;dataset=_dataset(training['inputs/features'].join(features),base[list(TARGETS)],training['inputs/common'],training['inputs/timing'],{'train':(original.spec.train_start,str(original.train_end.date())),'predict':(p['start'],p['end'])});auxiliary=_auxiliary_dataset(training['inputs/auxiliary']['outer'],str(original.train_end.date()))
state_path=card/'run_state.json';digest=hashlib.sha256((card/'plan.json').read_bytes()).hexdigest();state=json.loads(state_path.read_text()) if state_path.exists() else {'plan_sha256':digest,'fits':[],'native':[]};assert state['plan_sha256']==digest
kwargs={'experiment_name':card.name,'uri':str(card/'qlib_recorder')}
if state.get('recorder_id'):kwargs.update(recorder_id=state['recorder_id'],resume=True)
predictions={};audits=[];selection=[]
with R.start(**kwargs):
 rec=R.get_recorder();state.update(recorder_id=rec.id,experiment_id=str(rec.experiment_id));atomic_json(state,state_path);rec.save_objects(artifact_path='inputs',plan=p,source_audits={'training':ta,'source':sa,'controller':ca,'action':aa,'strong':sta,'eps':ea,'splits':spa},events=events,features=features,feature_manifest=manifest);rec.save_objects(artifact_path='sources',feature=(root/'src/data/delayed_seasonal_eps.py').read_text(),model=(root/'src/model/tangent_allocator.py').read_text(),base_model=(root/'src/model/minute_pit_allocator.py').read_text())
 for seed in p['seeds']:
  template=training[f'models/base_outer_{seed}'];models={f:TangentHeadAllocator(base_features=template.base_features+[FLAG],extra_features=[] if f=='control' else [FEATURE],feature_masks={**template.feature_masks,FEATURE:FLAG},spec={**asdict(template.spec),'root':str(root)},seed=seed,cutoff=str(template.cutoff),train_end=str(template.train_end.date()),minimum_observations=template.minimum_observations) for f in ['control','candidate']};initial=audit_zero_extended_heads(models['control']._initialize(),models['candidate']._initialize(),shared_primary_count=len(template.base_features)+1,extra_count=1)
  for family,model in models.items():
   name=f'{family}_{seed}'
   if name in state['fits']:model=rec.load_object('models/'+name);models[family]=model
   else:
    assert len(state['fits'])<p['maximum_model_fits'];model.fit(dataset,auxiliary_dataset=auxiliary);rec.save_objects(artifact_path='models',**{name:model});state['fits'].append(name);atomic_json(state,state_path)
   predictions[(family,seed)]=_prediction_dates(model.predict(dataset));audits.append({'family':family,'seed':seed,'initialization':initial,**model.audit});rec.log_metrics(completed_fits=len(state['fits']))
  a,b=models['control'],models['candidate'];assert a.primary_scaler==b.primary_scaler and a.common_scaler==b.common_scaler
  for key in ['training_rows','first_training_signal','last_training_signal','last_mature_label_at','auxiliary_rows','epochs']:assert a.audit[key]==b.audit[key]
  selection.append({'seed':seed,**b.extra_scaler[FEATURE]})
 rec.save_objects(artifact_path='analysis',predictions=predictions,training=audits,feature_usage=selection)
frame=source['inputs/frame'];frame=frame.loc[frame.signal_date.between(p['start'],p['end'])].copy();market=action['inputs/market'];dates=pd.DatetimeIndex(market.signal_date);returns=market.set_index('signal_date')[['soxl_return','soxx_return']];targets={f:_smooth_weights(_ensemble(predictions,f,p['seeds']).reindex(dates),spec=original.spec,candidate=original.candidate,asset_returns=returns) for f in ['control','candidate']}
recovery=IntradayRecoveryDirectAllocationSpec(root=root,**action['inputs/plan']['recovery_spec']);spec=Stage1OpenActionValueSpec(root=root,**action['inputs/plan']['action_spec']);books={}
for family in ['control','candidate']:
 local=frame.copy();aligned=targets[family].reindex(pd.DatetimeIndex(frame.signal_date));assert aligned.notna().all().all()
 for col in aligned:local[col]=aligned[col].to_numpy()
 for cost in p['slippage_bps']:
  key=f'{family}_{cost}'
  if key in state['native']:book=rec.load_object('analysis/native/'+key)
  else:
   assert len(state['native'])<p['maximum_new_native_cases'];replay=replay_full_path(local,models=None,plan=p,slippage_bps=cost,controller=controller['inputs/controller'],recovery_spec=recovery,action_spec=spec,selection=action['controller/source_recovery_selection']);book=native_tiered_stage1(local,replay,provider_uri=provider,tariff=p['tariff'],slippage_bps=cost,initial_cash=p['initial_cash'],freq='1min',opening_offset=pd.Timedelta(hours=9,minutes=47),position_type=QlibContinuousPosition);book['targets']=replay['targets'];rec.save_objects(artifact_path='analysis/native',**{key:book});state['native'].append(key);atomic_json(state,state_path)
  books[(family,cost)]=book;rec.log_metrics(completed_native_cases=len(state['native']))
for cost in p['slippage_bps']:
 for family,key in [('full_history_tangent','candidate'),('full_history_original','control'),('soxx','soxx')]:books[(family,cost)]=strong[f'analysis/native/{key}_{cost}'];rec.save_objects(artifact_path='analysis/native',**{f'{family}_{cost}':books[(family,cost)]})
rows=[];annual=[];uncertainty=[]
for (family,cost),book in books.items():
 path=book['authority'];b=books[('soxx',cost)]['authority'];assert path.trade_date.equals(b.trade_date);rel=np.log1p(path.strategy_return)-np.log1p(b.strategy_return);rows.append({'family':family,'slippage_bps':cost,**paired_metrics(path,b),'monitored_drawdown_pct':-100*path.monitored_drawdown.max(),'average_cash_pct':100*path.ending_cash_weight.mean(),'positive_126d_fraction':rel.rolling(126).sum().dropna().gt(0).mean()});annual.append(annual_paired_metrics(path,b).assign(family=family,slippage_bps=cost))
 if family=='candidate':
  for comparator in ['control','soxx','full_history_tangent']:uncertainty.append({'slippage_bps':cost,'comparator':comparator,**stationary_bootstrap_excess(path,books[(comparator,cost)]['authority'],mean_block=63,resamples=2000,alpha=.025,seed=20261015)})
events.to_csv(card/'event_scores.csv',index=False)
table=pd.DataFrame(rows);annual=pd.concat(annual,ignore_index=True);uncertainty=pd.DataFrame(uncertainty);cnd=table[table.family.eq('candidate')].set_index('slippage_bps');ctl=table[table.family.eq('control')].set_index('slippage_bps');stronger=table[table.family.eq('full_history_tangent')].set_index('slippage_bps');gates={'beat_paired_control_all_costs':bool(cnd.terminal_wealth.gt(ctl.terminal_wealth+1e-8).all()),'beat_strong_all_costs':bool(cnd.terminal_wealth.gt(stronger.terminal_wealth+1e-8).all()),'net_soxx_excess_all_costs':bool(cnd.relative_wealth.gt(1).all()),'drawdown_30pct':bool(cnd.max_drawdown_pct.ge(-30).all() and cnd.monitored_drawdown_pct.ge(-30).all()),'cvar_no_worse':bool(cnd.cvar5_pct.ge(cnd.benchmark_cvar5_pct).all()),'rolling_70pct':bool(cnd.positive_126d_fraction.ge(.7).all()),'annual_70pct':bool(annual[annual.family.eq('candidate')].groupby('slippage_bps').relative_wealth.apply(lambda x:x.gt(1).mean()).ge(.7).all()),'bootstrap_positive':bool(uncertainty.lower_bound_positive.all()),'independent_validation':False}
policy={'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','models/*','sources/*','analysis/predictions','analysis/native/*']};summary={'experiment':card.name,'decision':'needs_independent_validation' if all(v for k,v in gates.items() if k!='independent_validation') else p['failure_decision'],'gates':gates,'new_fits':len(state['fits']),'new_native_cases':len(state['native']),'provider_uri':str(provider),'region':'us','as_of':p['as_of'],'recorder_id':rec.id,'experiment_id':str(rec.experiment_id),'recorder_uri':str(card/'qlib_recorder'),'recorder_local_dir':str(card/'qlib_recorder'/str(rec.experiment_id)/rec.id),'artifact_policy':policy,'scope':p['scope'],'objective_achieved':False,'plan_sha256':digest,'feature_provider_uri':str(feature_provider),'conditional_source_timing':True,'production_qualified':False};rec.save_objects(artifact_path='analysis',summary=summary,comparison=table,annual=annual,uncertainty=uncertainty,targets=targets);rec.log_metrics(registered_artifact_count=sum(f.is_file() for f in (Path(summary['recorder_local_dir'])/'artifacts').rglob('*')))
for name,value in [('backtest_benchmark_comparison',table),('annual_metrics',annual),('uncertainty',uncertainty),('training_audit',pd.DataFrame(audits)),('feature_usage',pd.DataFrame(selection))]:value.to_csv(card/(name+'.csv'),index=False)
import yaml
(card/'workflow.yaml').write_text(yaml.safe_dump({'experiment':card.name,'provider_uri':str(provider),'segments':dataset.segments,'artifact_policy':policy}));a=Path(summary['recorder_local_dir'])/'artifacts';atomic_json({'recorder_id':rec.id,'entries':[{'path':str(f.relative_to(card)),'sha256':hashlib.sha256(f.read_bytes()).hexdigest()} for f in sorted(a.rglob('*')) if f.is_file()]},card/'verification_manifest.json');summary['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec.id,experiment_id=str(rec.experiment_id));atomic_json(summary,card/'summary.json');print(pd.DataFrame(selection).to_string(index=False));print(table[['family','slippage_bps','return_pct','monitored_drawdown_pct']].to_string(index=False))


# 延迟季节性盈利特征：真实训练回测，未通过强参考及SOXX

本轮完成6次新拟合、6套新增Qlib原生完整账户。双方均用原全历史主训练样本3014条（2011-01-03至2022-12-21）、20741辅助样本、3个原固定seed和300轮末轮等权集成；原标签、损失、平滑、控制器及风险预算不变。两组都增加相同可用性标记，候选仅增加一个数值字段。新增数值系数零初始化，初始输出逐值匹配；训练域与基础缩放器逐项相同。

特征为GAAP diluted EPS当季减去年同季，除以此前8个季度同比差的样本标准差（ddof=1，分母不含当前）。13个原报告值按当前报告时已知拆股换算到同一股份口径；后来会计可比值不回填。当前原报告值保留会计变化及一次性费用，不人为剔除Arm费用、出口管制费用或其他事后特殊季度。它是季节性代理，不是分析师一致预期或市场未预期利润。

时间用途与此前首发资料认证明确分开：每个事件取公告日、filed日期、API accepted原文日期和存在的index accepted原文日期的最晚一天，等该日结束，才在下一Qlib交易日使用；不推断两个accepted字段的时区互换。特征按实际trade_date对齐，再以原signal_date登记，持续至下一季度。此规则为条件性“延迟已申报信息重建”，不是首发瞬间信号或真实接收日志。父卡training_input_qualified=false没有被改写；本轮只检验该公开声明的保守重建假设下是否值得继续研究，不签发原HTML/首版字节/生产资格。SEC对filing date及晚间递交次日披露的说明支持分开处理，但不独立认证本轮每份文件的最早公开时刻。

48季度产生36个有限得分，首次可用2017-11-13、末次2026-08-28；每个seed有1287条已知训练观察，对应21个不同季度更新，日样本不能冒充独立事件。更早未知保持NaN，标记0；模型内部沿用训练内缩放及中性编码。所有新字段经独立Qlib store构建和D API读回，事件来源、拆股账本及时间假设均登记Recorder。

回测2023-01-03信号至2026-10-09退出，共944区间，10万美元，冻结IBKR Pro Tiered研究费率投影，0/2/5bp独立持仓、风控及计费路径。历史已反复使用，不是新鲜独立测试。

|账户|0bp净收益|2bp净收益|5bp净收益|
|---|---:|---:|---:|
|同可用标记对照|125.16%|123.82%|73.32%|
|新增季节性EPS|268.24%|291.05%|282.50%|
|原长期小配置头|342.52%|335.36%|348.06%|
|SOXX|387.75%|387.73%|387.70%|

候选2bp监控最大回撤29.86%、平均现金45.30%、126日滚动正超额41.27%。三个成本的回撤/CVaR及配对对照门通过；强参考、净SOXX超额、年度/滚动/联合区块区间门失败。可用标记改变了基础输入维度和初始化，匹配对照只能识别本配对中的数值增量；不能因击败这一较弱对照就说超过现有模型。原长期强参考始终保留。

决定not_promoted_delayed_seasonal_eps。关闭这一个固定季度代理/分母/日期滞后/模型组合，不改符号、窗口、滞后、特殊事件筛选或控制参数救结果。数据进入了训练，但没有支持替换现有强参考；即使将来补全首发时间认证，也不能单凭来源资格升级重开这次失败结果。下一需要独立的市场预期或多发行人信息增量，既有供应商surprise路线已失败，不能改名重跑。生产未改，稳定净SOXX超额未实现。

2项针对性测试通过：未来季度不影响过去、当前不进分母、拆股同比缩放不改变得分、日期缓冲严格后移及配对新增通道初始化。notebook内6模型/6账户执行完成、Qlib读回通过、NAS冷恢复逐对象哈希通过。真实分红、成交时延和全分钟最大回撤的原研究边界保留。

2bp年度候选 / SOXX：2023：34.98% / 56.60%；2024：27.53% / 20.97%；2025：47.43% / 46.80%；2026：54.08% / 75.38%。2026为不完整年度。

计划元数据说明：冻结计划继承的new_fits=3是未被执行器使用的旧描述字段；执行前已固定maximum_model_fits=6，mechanism/scope明确两组各3个seed，实际6次拟合符合该上限。保留原冻结计划及哈希，不按结果改计划；summary.new_fits=6为实际完成数。
